In [0]:
%run "../00. Common/Config"

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.drivers"
silver_table = f"{catalog_name}.{silver_schema}.drivers"


In [0]:
import pyspark.sql.functions as F

In [0]:
drivers_df = spark.table(bronze_table)

In [0]:
display(drivers_df)

In [0]:
drivers_selected_df = drivers_df.drop("url")

In [0]:
drivers_renamed_df = (
    drivers_selected_df
        .withColumnsRenamed({
            "driverId": "driver_id",
            "dateOfBirth": "date_of_birth"
        })
)

In [0]:
drivers_concat_df = (
    drivers_renamed_df
        .withColumn("driver_name",
                     F.concat_ws(" ", F.col("name.givenName"), F.col("name.familyName")))
        .drop("name")
)

In [0]:
display(drivers_concat_df)

In [0]:
drivers_distinct_df = drivers_concat_df.dropDuplicates(["driver_id"])

In [0]:
display(drivers_distinct_df)

In [0]:

drivers_final_df = (
    drivers_distinct_df
        .withColumn('nationality',F.initcap("nationality"))
        .withColumn('driver_name',F.initcap("driver_name"))
)

In [0]:
(
    drivers_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))